# Part C — Pandas and Exploratory Data Analysis

Dataset: UCI Student Performance (student-mat.csv). If the CSV is not present, the first code cell downloads it from the original UCI source. Run this notebook before Part D.

In [ ]:
from pathlib import Path
import urllib.request
import pandas as pd

DATA_FILE = Path("student-mat.csv")
DATA_URL = "https://huggingface.co/datasets/DElmazi/Student_Performance/resolve/main/student-mat.csv"

if not DATA_FILE.exists():
    print("Downloading the UCI Student Performance dataset...")
    urllib.request.urlretrieve(DATA_URL, DATA_FILE)

df = pd.read_csv(DATA_FILE, sep=";")

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)
display(df.head())

## Problem 8. Dataset inspection

In [ ]:
print("First five records:")
display(df.head())
print("Last five records:")
display(df.tail())
print("Rows and columns:", df.shape)
print("Column names:", df.columns.tolist())
print("Data types:")
print(df.dtypes)
print("Summary statistics:")
display(df.describe(include="all").T)

## Problem 9. Data cleaning

In [ ]:
print("Missing values before cleaning:")
print(df.isna().sum())
print("Duplicate rows:", df.duplicated().sum())
print("Data types before cleaning:")
print(df.dtypes)

numeric_candidates = [
    "age", "Medu", "Fedu", "traveltime", "studytime", "failures",
    "famrel", "freetime", "goout", "Dalc", "Walc", "health",
    "absences", "G1", "G2", "G3"
]
invalid_numeric_values = {}
for column in numeric_candidates:
    if column in df.columns:
        converted = pd.to_numeric(df[column], errors="coerce")
        invalid_count = int((converted.isna() & df[column].notna()).sum())
        if invalid_count:
            invalid_numeric_values[column] = invalid_count
print("Invalid values found in expected numeric columns:", invalid_numeric_values)

clean_df = df.copy()
numeric_columns = clean_df.select_dtypes(include="number").columns
categorical_columns = clean_df.select_dtypes(exclude="number").columns

for column in numeric_columns:
    clean_df[column] = pd.to_numeric(clean_df[column], errors="coerce")
    clean_df[column] = clean_df[column].fillna(clean_df[column].median())

for column in categorical_columns:
    clean_df[column] = clean_df[column].astype("string").str.strip()
    clean_df[column] = clean_df[column].replace("", pd.NA)
    if clean_df[column].isna().any():
        clean_df[column] = clean_df[column].fillna(clean_df[column].mode().iloc[0])

before = len(clean_df)
clean_df = clean_df.drop_duplicates().reset_index(drop=True)
print("Duplicate rows removed:", before - len(clean_df))
print("Missing values after cleaning:", int(clean_df.isna().sum().sum()))
print("Rows after cleaning:", len(clean_df))
print("Cleaning decisions: numeric missing values use the median; categorical missing values use the mode; duplicate rows are removed to avoid counting identical records twice.")

## Problem 10. Data analysis and visualization

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

print("Students with studytime >= 3 and absences < 10:")
display(clean_df[(clean_df["studytime"] >= 3) & (clean_df["absences"] < 10)].head())

print("Average final grade by study time:")
study_summary = clean_df.groupby("studytime")["G3"].agg(["mean", "count"]).round(2)
display(study_summary)

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
axes[0, 0].hist(clean_df["G3"], bins=15, edgecolor="black")
axes[0, 0].set_title("Distribution of final grades")
axes[0, 0].set_xlabel("Final grade (G3)")
axes[0, 0].set_ylabel("Number of students")

box_data = [clean_df.loc[clean_df["studytime"] == level, "G3"] for level in sorted(clean_df["studytime"].unique())]
box_labels = [str(level) for level in sorted(clean_df["studytime"].unique())]
axes[0, 1].boxplot(box_data, labels=box_labels)
axes[0, 1].set_title("Final grade by study time")
axes[0, 1].set_xlabel("Study time category")
axes[0, 1].set_ylabel("Final grade (G3)")

axes[1, 0].scatter(clean_df["absences"], clean_df["G3"], alpha=0.7)
axes[1, 0].set_title("Absences versus final grade")
axes[1, 0].set_xlabel("Absences")
axes[1, 0].set_ylabel("Final grade (G3)")

counts = clean_df["studytime"].value_counts().sort_index()
axes[1, 1].bar(counts.index.astype(str), counts.values, edgecolor="black")
axes[1, 1].set_title("Number of students by study time")
axes[1, 1].set_xlabel("Study time category")
axes[1, 1].set_ylabel("Number of students")
plt.tight_layout()
plt.savefig("eda_visualizations.png", dpi=150, bbox_inches="tight")
plt.show()

correlations = clean_df.select_dtypes(include="number").corr()
fig, ax = plt.subplots(figsize=(12, 9))
image = ax.imshow(correlations, cmap="coolwarm", vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(len(correlations.columns)), labels=correlations.columns, rotation=90)
ax.set_yticks(range(len(correlations.columns)), labels=correlations.columns)
ax.set_title("Correlation between numerical variables")
fig.colorbar(image, ax=ax, label="Correlation")
plt.tight_layout()
plt.savefig("eda_correlation.png", dpi=150, bbox_inches="tight")
plt.show()

grade_corr = correlations["G3"].drop("G3").sort_values(ascending=False)
print("Numerical correlations with final grade:")
display(grade_corr.to_frame("correlation"))

observations = [
    f"The dataset contains {len(clean_df)} records and {clean_df.shape[1]} columns after cleaning.",
    f"The average final grade (G3) is {clean_df['G3'].mean():.2f} out of 20.",
    f"The median final grade (G3) is {clean_df['G3'].median():.2f}.",
    f"The highest observed final grade is {clean_df['G3'].max():.0f} and the lowest is {clean_df['G3'].min():.0f}.",
    f"The numerical variable most positively correlated with G3, excluding G3 itself, is {grade_corr.index[0]} (correlation {grade_corr.iloc[0]:.2f}); correlation does not establish causation."
]
print("Five evidence-based observations:")
for number, observation in enumerate(observations, start=1):
    print(f"{number}. {observation}")
